In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    KFold,
    cross_validate,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    confusion_matrix,
)

In [ ]:
df = pd.read_csv("../data/cleaned/transformed.csv")

In [ ]:
X = df.drop(["churn"], axis=1)
Y = df["churn"]

In [ ]:
Y.value_counts()

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.3, random_state=3984
)
(x_train.shape, x_test.shape), (y_train.shape, y_test.shape)

In [ ]:
models = {
    "LogisticRegression": LogisticRegression(),
    "RandomForestClassifier": RandomForestClassifier(),
    "DecisionTreeClassifier": DecisionTreeClassifier(),
}

In [ ]:
cv = KFold(random_state=98347, n_splits=5, shuffle=True)

In [ ]:
models_evaluation = []

evaluation_metrics = ["accuracy", "precision", "recall"]
for model_name, estimator in models.items():
    scores = cross_validate(
        estimator, x_train, y_train, cv=cv, n_jobs=-1, scoring=evaluation_metrics
    )
    models_evaluation.append(
        {
            "model_name": model_name,
            "accuracy": scores["test_accuracy"].mean(),
            "precision": scores["test_precision"].mean(),
            "recall": scores["test_recall"].mean(),
        }
    )

In [ ]:
evaluation_df = pd.DataFrame(models_evaluation)
evaluation_df

In [ ]:
evaluation_df.plot(kind="bar")
plt.xticks([0, 1, 2], evaluation_df["model_name"].values, rotation=20)
plt.show()

### ***HyperParameterTune for best model***

In [ ]:
params = {
    "max_depth": [5, 10, 15],
    "n_estimators": [100, 200, 300],
    "min_samples_split": [4, 6, 10],
}

best_model = RandomizedSearchCV(
    estimator=RandomForestClassifier(n_jobs=-1),
    param_distributions=params,
    n_jobs=-1,
    cv=cv,
)

best_model.fit(x_train, y_train)

In [ ]:
best_model.best_params_

In [ ]:
random_forest_cl = RandomForestClassifier(**best_model.best_params_)

model = random_forest_cl.fit(x_train, y_train)

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
accuracy_score(y_pred, y_test) * 100